In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn torch

# A Federated Deep Learning Framework for Chronic Disease Complication Detection

## DisabilityCare-LIFE

MIMIC-III v1.4 · Stacked Autoencoder · Dwarf Mongoose Optimization · Inception-ResNet · GAN-Enhanced Federated Learning

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

## 1. Experimental Configuration

- Dataset: MIMIC-III Clinical Database, version 1.4
- Study cohort reported in the paper: 2,847 unique patients and 3,216 eligible admissions
- Model-ready observations reported in the paper: 5,000
- C1: cardiovascular complications
- C2: diabetic/metabolic complications
- C3: respiratory complications
- Stacked autoencoder encoder: 512, 256, and 128 neurons
- Sparse penalty coefficient: β = 0.001
- Inception-ResNet: four blocks with parallel 1×1, 1×3, and 1×5 convolutions
- Dwarf Mongoose Optimization: 30 agents and 500 iterations
- Federated learning: 10 simulated clients and five aggregation rounds
- Evaluation: five-fold patient-level cross-validation
- Metrics: accuracy, precision, recall, F1-score, MCC, and false-alarm rate

The manuscript does not enumerate the ICD-9-CM code list, sensor-to-MIMIC variable mapping, disability-specific scaling bounds, DMO search bounds, or differential-privacy parameters. Cohort selection below therefore uses diagnosis-description terms for the disability and complication categories stated in the manuscript. MIMIC-III is an ICU database and does not contain the complete wearable EMG, IMU, and assistive-device sensor streams described for prospective deployment.

In [ ]:
import gc
import json
import math
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix, classification_report
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. MIMIC-III Dataset

Tables used for cohort construction and clinical feature extraction:

- `PATIENTS.csv`
- `ADMISSIONS.csv`
- `DIAGNOSES_ICD.csv`
- `D_ICD_DIAGNOSES.csv`
- `CHARTEVENTS.csv`
- `LABEVENTS.csv`

MIMIC-III access is governed by the PhysioNet credentialed research-access procedure and Data Use Agreement.

In [ ]:
def locate_mimic_directory():
    candidates = [
        Path.cwd() / "mimic-iii-clinical-database-1.4",
        Path.cwd() / "mimic-iii",
        Path("/content/mimic-iii-clinical-database-1.4"),
        Path("/content/drive/MyDrive/mimic-iii-clinical-database-1.4"),
        Path("/content/drive/MyDrive/mimic-iii"),
        Path("/mnt/data/mimic-iii-clinical-database-1.4")
    ]
    required = [
        "PATIENTS.csv", "ADMISSIONS.csv", "DIAGNOSES_ICD.csv",
        "D_ICD_DIAGNOSES.csv", "CHARTEVENTS.csv", "LABEVENTS.csv"
    ]
    for root in candidates:
        if root.exists() and all((root / f).exists() for f in required):
            return root
    for base in [Path.cwd(), Path("/content"), Path("/mnt/data")]:
        if base.exists():
            for p in base.glob("**/PATIENTS.csv"):
                if all((p.parent / f).exists() for f in required):
                    return p.parent
    raise FileNotFoundError(
        "MIMIC-III v1.4 CSV files were not found. The extracted database directory "
        "must contain PATIENTS.csv, ADMISSIONS.csv, DIAGNOSES_ICD.csv, "
        "D_ICD_DIAGNOSES.csv, CHARTEVENTS.csv, and LABEVENTS.csv."
    )

MIMIC_DIR = locate_mimic_directory()
print("MIMIC-III directory:", MIMIC_DIR)

## 3. Cohort Definition and Leakage Prevention

Admission diagnoses are linked through `HADM_ID`; patient identity is retained through `SUBJECT_ID`. Eligible admissions must contain a physical-disability diagnosis and at least one target complication category. Patient identity is retained for all data partitions and federated-client assignments.

In [ ]:
patients = pd.read_csv(MIMIC_DIR / "PATIENTS.csv")
admissions = pd.read_csv(MIMIC_DIR / "ADMISSIONS.csv")
diagnoses = pd.read_csv(MIMIC_DIR / "DIAGNOSES_ICD.csv", dtype={"ICD9_CODE": str})
diagnosis_dictionary = pd.read_csv(
    MIMIC_DIR / "D_ICD_DIAGNOSES.csv", dtype={"ICD9_CODE": str}
)

diagnoses = diagnoses.merge(
    diagnosis_dictionary[["ICD9_CODE", "SHORT_TITLE", "LONG_TITLE"]],
    on="ICD9_CODE", how="left"
)
diagnoses["DIAGNOSIS_TEXT"] = (
    diagnoses["SHORT_TITLE"].fillna("") + " " +
    diagnoses["LONG_TITLE"].fillna("")
).str.lower()

disability_terms = (
    r"spinal cord injur|parapleg|quadripleg|tetrapleg|cerebral pals|"
    r"muscular dystroph|multiple sclerosis|physical disabil"
)
target_patterns = {
    0: r"cardiac|cardiovascular|heart failure|myocardial|ischemic heart|arrhythmia|hypertens|vascular disease",
    1: r"diabetes|diabetic|hyperglyc|hypoglyc|metabolic",
    2: r"respiratory failure|respiratory distress|pneumonia|chronic obstructive pulmonary|pulmonary disease|atelectasis"
}
diagnoses["is_disability"] = diagnoses["DIAGNOSIS_TEXT"].str.contains(
    disability_terms, regex=True, na=False
)
diagnoses["class_id"] = -1
for class_id, pattern in target_patterns.items():
    mask = diagnoses["DIAGNOSIS_TEXT"].str.contains(pattern, regex=True, na=False)
    diagnoses.loc[mask & (diagnoses["class_id"] == -1), "class_id"] = class_id

admission_flags = diagnoses.groupby("HADM_ID").agg(
    has_disability=("is_disability", "max"),
    has_target=("class_id", lambda s: (s >= 0).any())
).reset_index()
eligible_ids = admission_flags.loc[
    admission_flags["has_disability"] & admission_flags["has_target"], "HADM_ID"
]
primary_class = (
    diagnoses[(diagnoses["HADM_ID"].isin(eligible_ids)) & (diagnoses["class_id"] >= 0)]
    .sort_values(["HADM_ID", "SEQ_NUM"])
    .drop_duplicates("HADM_ID")[["HADM_ID", "class_id"]]
)
cohort = (
    admissions.merge(primary_class, on="HADM_ID", how="inner")
    .merge(patients[["SUBJECT_ID", "GENDER", "DOB"]], on="SUBJECT_ID", how="inner")
)
cohort["ADMITTIME"] = pd.to_datetime(cohort["ADMITTIME"], errors="coerce")
cohort["DOB"] = pd.to_datetime(cohort["DOB"], errors="coerce")
cohort["AGE"] = ((cohort["ADMITTIME"] - cohort["DOB"]).dt.days / 365.25).clip(upper=90)
cohort = cohort.dropna(subset=["SUBJECT_ID", "HADM_ID", "ADMITTIME", "AGE"])
cohort["class_id"] = cohort["class_id"].astype(int)

print("Eligible unique patients:", cohort["SUBJECT_ID"].nunique())
print("Eligible admissions:", cohort["HADM_ID"].nunique())
print(cohort["class_id"].value_counts().sort_index())

## 4. Clinical Variable Extraction

Chart and laboratory measurements are aligned by admission and hour. The extracted variables provide MIMIC-III clinical proxies for cardiovascular, respiratory, and metabolic monitoring. Continuous wearable ECG-derived HRV, EMG, and IMU streams are not available in MIMIC-III.

In [ ]:
CHART_ITEM_MAP = {
    211: "heart_rate", 220045: "heart_rate",
    51: "systolic_bp", 442: "systolic_bp", 455: "systolic_bp",
    6701: "systolic_bp", 220179: "systolic_bp", 220050: "systolic_bp",
    8368: "diastolic_bp", 8440: "diastolic_bp", 8441: "diastolic_bp",
    8555: "diastolic_bp", 220180: "diastolic_bp", 220051: "diastolic_bp",
    618: "respiratory_rate", 615: "respiratory_rate",
    220210: "respiratory_rate", 224690: "respiratory_rate",
    646: "oxygen_saturation", 220277: "oxygen_saturation",
    223762: "temperature_c", 678: "temperature_c", 223761: "temperature_c"
}
LAB_ITEM_MAP = {
    50809: "glucose", 50931: "glucose", 807: "glucose",
    811: "glucose", 1529: "glucose", 3745: "glucose",
    50813: "lactate", 50814: "lactate",
    50902: "chloride", 50912: "creatinine",
    50983: "sodium", 50824: "sodium",
    50822: "potassium", 50971: "potassium",
    51301: "wbc", 51300: "wbc",
    51222: "hemoglobin", 50811: "hemoglobin"
}

chart_frames = []
for chunk in pd.read_csv(
    MIMIC_DIR / "CHARTEVENTS.csv",
    usecols=["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUENUM"],
    chunksize=1_000_000
):
    chunk = chunk[
        chunk["ITEMID"].isin(CHART_ITEM_MAP) &
        chunk["HADM_ID"].isin(cohort["HADM_ID"])
    ].copy()
    if len(chunk):
        chunk["feature"] = chunk["ITEMID"].map(CHART_ITEM_MAP)
        chart_frames.append(chunk)

lab_frames = []
for chunk in pd.read_csv(
    MIMIC_DIR / "LABEVENTS.csv",
    usecols=["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUENUM"],
    chunksize=1_000_000
):
    chunk = chunk[
        chunk["ITEMID"].isin(LAB_ITEM_MAP) &
        chunk["HADM_ID"].isin(cohort["HADM_ID"])
    ].copy()
    if len(chunk):
        chunk["feature"] = chunk["ITEMID"].map(LAB_ITEM_MAP)
        lab_frames.append(chunk)

chart_events = pd.concat(chart_frames, ignore_index=True) if chart_frames else pd.DataFrame()
lab_events = pd.concat(lab_frames, ignore_index=True) if lab_frames else pd.DataFrame()
events = pd.concat([chart_events, lab_events], ignore_index=True)
events["CHARTTIME"] = pd.to_datetime(events["CHARTTIME"], errors="coerce")
events["VALUENUM"] = pd.to_numeric(events["VALUENUM"], errors="coerce")
events = events.dropna(subset=["HADM_ID", "CHARTTIME", "VALUENUM", "feature"])
events = events.merge(
    cohort[["SUBJECT_ID", "HADM_ID", "ADMITTIME", "class_id", "AGE", "GENDER"]],
    on=["SUBJECT_ID", "HADM_ID"], how="inner"
)
events["hour"] = (
    (events["CHARTTIME"] - events["ADMITTIME"]).dt.total_seconds() // 3600
).astype(int)
events = events[(events["hour"] >= 0) & (events["hour"] < 72)]

hourly = events.groupby(
    ["SUBJECT_ID", "HADM_ID", "hour", "feature"], observed=True
)["VALUENUM"].median().unstack("feature").reset_index()
metadata = cohort[["SUBJECT_ID", "HADM_ID", "class_id", "AGE", "GENDER"]].drop_duplicates("HADM_ID")
hourly = hourly.merge(metadata, on=["SUBJECT_ID", "HADM_ID"], how="inner")
hourly["gender_code"] = hourly["GENDER"].map({"M": 1.0, "F": 0.0})
FEATURES = [
    "heart_rate", "systolic_bp", "diastolic_bp", "respiratory_rate",
    "oxygen_saturation", "temperature_c", "glucose", "lactate",
    "chloride", "creatinine", "sodium", "potassium", "wbc",
    "hemoglobin", "AGE", "gender_code"
]
FEATURES = [c for c in FEATURES if c in hourly.columns]
clinical_features = [c for c in FEATURES if c not in ["AGE", "gender_code"]]
hourly = hourly[hourly[clinical_features].notna().sum(axis=1) >= 3].copy()
hourly = hourly.sort_values(["SUBJECT_ID", "HADM_ID", "hour"]).reset_index(drop=True)
hourly["sample_id"] = np.arange(len(hourly))
print("Hourly analytical observations:", len(hourly))
print("Unique patients:", hourly["SUBJECT_ID"].nunique())
print("Features:", FEATURES)
hourly.head()

## 5. Edge Preprocessing

Channel-wise Z-score normalization is calculated within each admission using expanding statistics. Missing values are imputed using training-partition medians, and feature scaling is fitted only on the training partition.

In [ ]:
def edge_zscore(frame, columns):
    out = frame.copy()
    for col in columns:
        grouped = out.groupby("HADM_ID")[col]
        mean = grouped.transform(lambda s: s.expanding(min_periods=1).mean())
        std = grouped.transform(lambda s: s.expanding(min_periods=2).std())
        std = std.replace(0, np.nan)
        out[col] = ((out[col] - mean) / std).replace([np.inf, -np.inf], np.nan)
    return out

hourly = edge_zscore(hourly, FEATURES)
X_raw = hourly[FEATURES].copy()
y = hourly["class_id"].to_numpy(dtype=np.int64)
groups = hourly["SUBJECT_ID"].to_numpy()

## 6. Stacked Autoencoder

The encoder dimensions are 512, 256, and 128, with ReLU activations and β = 0.001 sparsity regularization.

In [ ]:
class StackedAutoencoder(nn.Module):
    def __init__(self, input_dim, beta=0.001, target_sparsity=0.05):
        super().__init__()
        self.beta = beta
        self.target_sparsity = target_sparsity
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def loss(self, x, reconstruction, z):
        reconstruction_loss = F.mse_loss(reconstruction, x)
        rho = torch.full((z.shape[1],), self.target_sparsity, device=z.device)
        rho_hat = torch.clamp(z.mean(0), 1e-6, 1 - 1e-6)
        sparse_loss = torch.sum(
            rho * torch.log(rho / rho_hat) +
            (1 - rho) * torch.log((1 - rho) / (1 - rho_hat))
        )
        return reconstruction_loss + self.beta * sparse_loss

class FeatureDataset(Dataset):
    def __init__(self, x):
        self.x = torch.as_tensor(x, dtype=torch.float32)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, index):
        return self.x[index]

## 7. Dwarf Mongoose Optimization-Based Inception Residual Network

Four Inception residual blocks process parallel 1×1, 1×3, and 1×5 convolution branches. Global average pooling and a Softmax classifier produce probabilities for C1, C2, and C3.

In [ ]:
class InceptionResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        b = channels // 3
        self.branch1 = nn.Sequential(
            nn.Conv1d(channels, b, 1, bias=False),
            nn.BatchNorm1d(b), nn.ReLU()
        )
        self.branch3 = nn.Sequential(
            nn.Conv1d(channels, b, 3, padding=1, bias=False),
            nn.BatchNorm1d(b), nn.ReLU()
        )
        self.branch5 = nn.Sequential(
            nn.Conv1d(channels, channels - 2*b, 5, padding=2, bias=False),
            nn.BatchNorm1d(channels - 2*b), nn.ReLU()
        )
        self.project = nn.Sequential(
            nn.Conv1d(channels, channels, 1, bias=False),
            nn.BatchNorm1d(channels)
        )
        self.activation = nn.ReLU()

    def forward(self, x):
        y = torch.cat([self.branch1(x), self.branch3(x), self.branch5(x)], dim=1)
        return self.activation(self.project(y) + x)

class DMIRNetwork(nn.Module):
    def __init__(self, num_classes=3, dropout=0.2):
        super().__init__()
        channels = 64
        self.stem = nn.Sequential(
            nn.Conv1d(1, channels, 1, bias=False),
            nn.BatchNorm1d(channels), nn.ReLU()
        )
        self.blocks = nn.Sequential(*[InceptionResidualBlock(channels) for _ in range(4)])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(channels, num_classes)

    def forward(self, z):
        x = self.stem(z.unsqueeze(1))
        x = self.blocks(x)
        x = F.adaptive_avg_pool1d(x, 1).squeeze(-1)
        return self.classifier(self.dropout(x))

class DisabilityCareModel(nn.Module):
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.sae = StackedAutoencoder(input_dim, beta=0.001)
        self.classifier = DMIRNetwork(num_classes=3, dropout=dropout)

    def forward(self, x):
        reconstruction, z = self.sae(x)
        logits = self.classifier(z)
        return logits, reconstruction, z

## 8. Dwarf Mongoose Optimization

The manuscript specifies 30 agents and 500 iterations and describes a fitness function balancing classification accuracy, clinical safety, and responsiveness. The numerical search bounds and complete fitness equation are not provided in the manuscript text. The optimizer below searches the three class decision thresholds on validation probabilities; its objective combines accuracy, macro-F1, and one-vs-rest false-alarm rate.

In [ ]:
def false_alarm_rate(y_true, y_pred, num_classes=3):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
    rates = []
    for c in range(num_classes):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - fn - fp
        rates.append(fp / (fp + tn) if fp + tn else 0.0)
    return float(np.mean(rates))

def threshold_predictions(probabilities, thresholds):
    return (probabilities / np.maximum(np.asarray(thresholds)[None, :], 1e-6)).argmax(axis=1)

def dmo_threshold_search(y_true, probabilities, agents=30, iterations=500, seed=42):
    rng = np.random.default_rng(seed)
    positions = rng.uniform(0.35, 0.85, size=(agents, 3))

    def fitness(position):
        pred = threshold_predictions(probabilities, position)
        acc = accuracy_score(y_true, pred)
        mf1 = f1_score(y_true, pred, average="macro", zero_division=0)
        far = false_alarm_rate(y_true, pred)
        return 0.4 * acc + 0.4 * mf1 + 0.2 * (1 - far)

    scores = np.array([fitness(p) for p in positions])
    best_i = int(scores.argmax())
    best_position = positions[best_i].copy()
    best_score = float(scores[best_i])
    history = [best_score]

    for _ in range(iterations):
        alpha = positions[int(scores.argmax())].copy()
        volatility = positions.std(axis=0) + 1e-8
        proposals = positions + rng.uniform(-1, 1, positions.shape) * (
            alpha - positions
        ) * rng.random((agents, 1))
        scout = rng.random(agents) < 0.25
        proposals[scout] += rng.normal(0, 1, (scout.sum(), 3)) * volatility
        proposals += rng.uniform(-0.1, 0.1, positions.shape) * (
            best_position - positions
        )
        proposals = np.clip(proposals, 0.35, 0.85)
        new_scores = np.array([fitness(p) for p in proposals])
        improved = new_scores > scores
        positions[improved] = proposals[improved]
        scores[improved] = new_scores[improved]
        if scores.max() > best_score:
            best_i = int(scores.argmax())
            best_position = positions[best_i].copy()
            best_score = float(scores[best_i])
        history.append(best_score)
    return best_position, best_score, history

## 9. GAN-Enhanced Federated Learning

Each simulated client trains a conditional generator and discriminator using local clinical features. Generated feature vectors and complication labels are shared for synthetic augmentation. Raw client observations remain within their assigned client partitions.

In [ ]:
class ConditionalGenerator(nn.Module):
    def __init__(self, input_dim, noise_dim=32, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, 8)
        self.network = nn.Sequential(
            nn.Linear(noise_dim + 8, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, input_dim)
        )
    def forward(self, noise, labels):
        return self.network(torch.cat([noise, self.embedding(labels)], dim=1))

class ConditionalDiscriminator(nn.Module):
    def __init__(self, input_dim, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, 8)
        self.network = nn.Sequential(
            nn.Linear(input_dim + 8, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x, labels):
        return self.network(torch.cat([x, self.embedding(labels)], dim=1))

def train_client_gan(x, y, input_dim, epochs=3, batch_size=128):
    generator = ConditionalGenerator(input_dim).to(DEVICE)
    discriminator = ConditionalDiscriminator(input_dim).to(DEVICE)
    opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    ds = torch.utils.data.TensorDataset(
        torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    bce = nn.BCEWithLogitsLoss()

    for _ in range(epochs):
        for real, labels in loader:
            real, labels = real.to(DEVICE), labels.to(DEVICE)
            n = len(real)
            ones = torch.ones(n, 1, device=DEVICE)
            zeros = torch.zeros(n, 1, device=DEVICE)
            fake = generator(torch.randn(n, 32, device=DEVICE), labels).detach()
            loss_d = bce(discriminator(real, labels), ones) + bce(discriminator(fake, labels), zeros)
            opt_d.zero_grad()
            loss_d.backward()
            opt_d.step()
            generated = generator(torch.randn(n, 32, device=DEVICE), labels)
            loss_g = bce(discriminator(generated, labels), ones)
            opt_g.zero_grad()
            loss_g.backward()
            opt_g.step()
    return generator

@torch.no_grad()
def generate_synthetic(generator, counts, input_dim, per_class_limit=100):
    generator.eval()
    xs, ys = [], []
    for class_id, count in enumerate(counts):
        n = min(int(count), per_class_limit)
        if n:
            labels = torch.full((n,), class_id, dtype=torch.long, device=DEVICE)
            samples = generator(torch.randn(n, 32, device=DEVICE), labels).cpu().numpy()
            xs.append(samples)
            ys.append(np.full(n, class_id, dtype=np.int64))
    if not xs:
        return np.empty((0, input_dim), np.float32), np.empty(0, np.int64)
    return np.vstack(xs).astype(np.float32), np.concatenate(ys)

## 10. Federated Training

The non-IID environment uses 10 simulated clients. Patient groups are assigned to one client only. Local gradients are clipped before optimization, and Gaussian noise is applied to aggregated model updates. Five global aggregation rounds are performed.

In [ ]:
NUM_CLIENTS = 10
FED_ROUNDS = 5
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
GRADIENT_CLIP_NORM = 1.0
DP_NOISE_MULTIPLIER = 0.001

def make_patient_clients(x, y, patient_groups, num_clients=10, seed=42):
    unique_patients = np.unique(patient_groups)
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_patients)
    clients = []
    for ids in np.array_split(unique_patients, num_clients):
        mask = np.isin(patient_groups, ids)
        if mask.any():
            clients.append((x[mask], y[mask], patient_groups[mask]))
    return clients

def state_copy(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def train_local_model(global_state, x, y, input_dim, synthetic_x=None, synthetic_y=None):
    model = DisabilityCareModel(input_dim).to(DEVICE)
    model.load_state_dict(global_state)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    if synthetic_x is not None and len(synthetic_x):
        x = np.vstack([x, synthetic_x])
        y = np.concatenate([y, synthetic_y])
    ds = torch.utils.data.TensorDataset(
        torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)
    )
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    model.train()
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits, reconstruction, z = model(xb)
            loss = F.cross_entropy(logits, yb) + model.sae.loss(xb, reconstruction, z)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
            optimizer.step()
    return state_copy(model)

def federated_average(states, sizes):
    total = float(sum(sizes))
    result = {}
    for key in states[0]:
        aggregate = torch.stack([
            state[key].float() * (size / total)
            for state, size in zip(states, sizes)
        ]).sum(0)
        if aggregate.is_floating_point() and DP_NOISE_MULTIPLIER:
            aggregate += torch.randn_like(aggregate) * DP_NOISE_MULTIPLIER
        result[key] = aggregate
    return result

def train_federated(clients, input_dim):
    global_model = DisabilityCareModel(input_dim).to(DEVICE)
    global_state = state_copy(global_model)
    communication_mb = []
    for _ in range(FED_ROUNDS):
        generators = [
            train_client_gan(xc, yc, input_dim)
            for xc, yc, _ in clients
        ]
        synthetic_sets = [
            generate_synthetic(
                gen, np.bincount(yc, minlength=3), input_dim
            )
            for gen, (xc, yc, _) in zip(generators, clients)
        ]
        nonempty_sets = [s for s in synthetic_sets if len(s[0])]
        if nonempty_sets:
            synthetic_x = np.vstack([s[0] for s in nonempty_sets])
            synthetic_y = np.concatenate([s[1] for s in nonempty_sets])
        else:
            synthetic_x = np.empty((0, input_dim), np.float32)
            synthetic_y = np.empty(0, np.int64)

        local_states, sizes = [], []
        for xc, yc, _ in clients:
            local_states.append(train_local_model(
                global_state, xc, yc, input_dim, synthetic_x, synthetic_y
            ))
            sizes.append(len(yc))
        global_state = federated_average(local_states, sizes)
        global_model.load_state_dict(global_state)
        parameter_bytes = sum(v.numel() * v.element_size() for v in global_state.values())
        communication_mb.append(2 * NUM_CLIENTS * parameter_bytes / (1024 ** 2))
        del generators
        gc.collect()
    return global_model, communication_mb

## 11. Patient-Level Five-Fold Cross-Validation

Stratified group folds keep all observations from each patient in a single fold. A patient-grouped validation split is used for DMO threshold optimization.

In [ ]:
def prepare_fold(train_idx, test_idx):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    train_rel, val_rel = next(
        splitter.split(train_idx, y[train_idx], groups=groups[train_idx])
    )
    train_rows, val_rows = train_idx[train_rel], train_idx[val_rel]
    medians = X_raw.iloc[train_rows].median()
    x_train = X_raw.iloc[train_rows].fillna(medians).to_numpy(np.float32)
    x_val = X_raw.iloc[val_rows].fillna(medians).to_numpy(np.float32)
    x_test = X_raw.iloc[test_idx].fillna(medians).to_numpy(np.float32)
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train).astype(np.float32)
    x_val = scaler.transform(x_val).astype(np.float32)
    x_test = scaler.transform(x_test).astype(np.float32)
    return (
        x_train, y[train_rows], groups[train_rows],
        x_val, y[val_rows], x_test, y[test_idx],
        train_rows, val_rows
    )

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(X_raw, y, groups))
print("Patient-level folds:", len(folds))

## 12. Model Training and Evaluation

In [ ]:
def predict_probabilities(model, x, batch_size=512):
    model.eval()
    loader = DataLoader(FeatureDataset(x), batch_size=batch_size, shuffle=False)
    predictions = []
    with torch.no_grad():
        for xb in loader:
            logits, _, _ = model(xb.to(DEVICE))
            predictions.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(predictions)

def classification_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-score": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "FAR": false_alarm_rate(y_true, y_pred)
    }

fold_results, fold_predictions, fold_models, fold_communication = [], [], [], []
for fold_no, (train_idx, test_idx) in enumerate(folds, 1):
    print(f"Fold {fold_no}/5")
    x_train, y_train, g_train, x_val, y_val, x_test, y_test, train_rows, val_rows = prepare_fold(train_idx, test_idx)
    clients = make_patient_clients(x_train, y_train, g_train, NUM_CLIENTS, SEED + fold_no)
    model, communication = train_federated(clients, x_train.shape[1])
    val_prob = predict_probabilities(model, x_val)
    thresholds, dmo_fitness, dmo_history = dmo_threshold_search(
        y_val, val_prob, agents=30, iterations=500, seed=SEED + fold_no
    )
    test_prob = predict_probabilities(model, x_test)
    test_pred = threshold_predictions(test_prob, thresholds)
    metrics = classification_metrics(y_test, test_pred)
    metrics.update({
        "Fold": fold_no, "DMO_Fitness": dmo_fitness,
        "Threshold_C1": thresholds[0], "Threshold_C2": thresholds[1],
        "Threshold_C3": thresholds[2]
    })
    fold_results.append(metrics)
    fold_predictions.append((y_test, test_pred, test_prob))
    fold_models.append(model.cpu())
    fold_communication.append(communication)
    print(metrics)
    del model, clients
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(fold_results)
results_df

## 13. Overall Classification Performance

In [ ]:
metric_columns = ["Accuracy", "Precision", "Recall", "F1-score", "MCC", "FAR"]
summary = results_df[metric_columns].agg(["mean", "std"]).T
summary

In [ ]:
all_true = np.concatenate([v[0] for v in fold_predictions])
all_pred = np.concatenate([v[1] for v in fold_predictions])
print(classification_report(
    all_true, all_pred,
    target_names=["C1 Cardiovascular", "C2 Diabetic/metabolic", "C3 Respiratory"],
    digits=4, zero_division=0
))
print(classification_metrics(all_true, all_pred))

cm = confusion_matrix(all_true, all_pred, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
labels = ["C1 Cardiovascular", "C2 Diabetic/metabolic", "C3 Respiratory"]
ax.set_xticks(range(3), labels, rotation=20, ha="right")
ax.set_yticks(range(3), labels)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("DisabilityCare-LIFE Confusion Matrix")
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 14. Matthews Correlation Coefficient Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(["DisabilityCare-LIFE"], [results_df["MCC"].mean()])
ax.set_ylim(0, 1)
ax.set_ylabel("Matthews correlation coefficient")
ax.set_title("Matthews Correlation Coefficient")
ax.text(0, results_df["MCC"].mean() + 0.02, f'{results_df["MCC"].mean():.4f}', ha="center")
plt.tight_layout()
plt.show()

## 15. False Alarm Rate Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(results_df["Fold"], results_df["FAR"] * 100, marker="o", linewidth=2)
ax.set_xlabel("Cross-validation fold")
ax.set_ylabel("False alarm rate (%)")
ax.set_title("False Alarm Rate")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 16. Communication Cost Analysis

In [ ]:
communication_df = pd.DataFrame({
    "Fold": np.arange(1, len(fold_communication) + 1),
    "Communication cost (MB)": [float(np.mean(v)) for v in fold_communication]
})
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(communication_df["Fold"], communication_df["Communication cost (MB)"], marker="o")
ax.set_xlabel("Cross-validation fold")
ax.set_ylabel("Communication cost (MB)")
ax.set_title("Federated Communication Cost")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
communication_df

## 17. Throughput and Latency Analysis

In [ ]:
inference_model = fold_models[-1].to(DEVICE).eval()
benchmark_x = torch.tensor(x_test[:min(512, len(x_test))], dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    for _ in range(10):
        inference_model(benchmark_x)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
start = time.perf_counter()
repetitions = 100
with torch.no_grad():
    for _ in range(repetitions):
        inference_model(benchmark_x)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start
throughput_tps = repetitions * len(benchmark_x) / elapsed
latency_seconds = elapsed / repetitions
system_metrics = pd.DataFrame({
    "Metric": ["Throughput (transactions/s)", "Latency (s per batch)"],
    "Measured value": [throughput_tps, latency_seconds]
})
system_metrics

## 18. Comparative Classification Performance

The following values are the classification results reported in the paper. They are distinct from the metrics measured by this notebook.

In [ ]:
reported_comparison = pd.DataFrame({
    "Model": ["DisabilityCare-LIFE", "LLM-based Chatbot", "PHMS-Fed", "EDCNNS"],
    "Accuracy": [98.7, 96.6, 95.8, 94.8],
    "Precision": [98.4, 96.2, 95.3, 94.1],
    "Recall": [98.6, 96.4, 95.6, 94.5],
    "F1-score": [98.5, 96.3, 95.4, 94.3],
    "Macro-average": [98.55, 96.38, 95.53, 94.43]
})
reported_comparison

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-score"]
x = np.arange(len(reported_comparison))
width = 0.19
fig, ax = plt.subplots(figsize=(11, 6))
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + (i - 1.5) * width, reported_comparison[metric], width, label=metric)
ax.set_xticks(x, reported_comparison["Model"], rotation=12, ha="right")
ax.set_ylabel("Performance (%)")
ax.set_ylim(85, 100)
ax.set_title("Classification Performance Reported in the Paper")
ax.legend(ncol=4)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 19. Non-IID Federated Client Evaluation

In [ ]:
client_rows = []
for client_id, (client_x, client_y, client_groups) in enumerate(
    make_patient_clients(x_train, y_train, g_train, NUM_CLIENTS, SEED), 1
):
    client_rows.append({
        "Client": client_id,
        "Patients": np.unique(client_groups).size,
        "Observations": len(client_y),
        "C1": int((client_y == 0).sum()),
        "C2": int((client_y == 1).sum()),
        "C3": int((client_y == 2).sum())
    })
client_distribution = pd.DataFrame(client_rows)
client_distribution

## 20. Ablation Analysis

The following values are the incremental ablation results reported in the paper; they are not recalculated from this MIMIC-III execution.

In [ ]:
ablation_reported = pd.DataFrame({
    "Configuration": [
        "Baseline temporal model", "+ DATQM", "+ PFTA", "+ FSDG",
        "+ Privacy mechanisms", "Full DisabilityCare-LIFE"
    ],
    "Accuracy": [95.2, 96.1, 97.1, 98.0, 98.4, 98.7],
    "Precision": [94.8, 95.7, 96.8, 97.7, 98.1, 98.4],
    "Recall": [95.0, 96.0, 97.0, 97.9, 98.3, 98.6],
    "F1-score": [94.9, 95.8, 96.9, 97.8, 98.2, 98.5],
    "MCC": [0.918, 0.932, 0.947, 0.960, 0.967, 0.972]
})
ablation_reported

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for metric in ["Accuracy", "Precision", "Recall", "F1-score"]:
    ax.plot(ablation_reported["Configuration"], ablation_reported[metric],
            marker="o", linewidth=2, label=metric)
ax.set_ylabel("Performance (%)")
ax.set_title("Ablation Analysis Reported in the Paper")
ax.tick_params(axis="x", rotation=18)
ax.legend(ncol=4)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 21. Reported System-Level Comparison

The values below are the system-level results stated in the manuscript. Notebook-measured communication, throughput, and latency are presented separately.

In [ ]:
reported_system = pd.DataFrame({
    "Metric": [
        "MCC", "False alarm rate at epoch 100", "Throughput (tps)",
        "Latency (s)", "Communication cost (MB)"
    ],
    "DisabilityCare-LIFE": [0.972, 1.2, 125, 2.1, 48],
    "Best baseline": [0.958, 4.6, 104, 3.2, 63]
})
reported_system

## 22. Experimental Results

In [ ]:
measured_results = {
    "cross_validation": results_df.to_dict(orient="records"),
    "mean_metrics": summary["mean"].to_dict(),
    "standard_deviation": summary["std"].to_dict(),
    "system_measurements": system_metrics.to_dict(orient="records"),
    "communication_cost": communication_df.to_dict(orient="records"),
    "class_names": ["C1 Cardiovascular", "C2 Diabetic/metabolic", "C3 Respiratory"],
    "feature_columns": FEATURES,
    "eligible_patients": int(cohort["SUBJECT_ID"].nunique()),
    "eligible_admissions": int(cohort["HADM_ID"].nunique()),
    "hourly_observations": int(len(hourly))
}
print(json.dumps(measured_results, indent=2))

## 23. Conclusion

DisabilityCare-LIFE integrates physiological monitoring, edge preprocessing, stacked-autoencoder feature extraction, Dwarf Mongoose optimization-based Inception-ResNet classification, and GAN-enhanced federated learning. MIMIC-III evaluates the retrospective clinical prediction component. Prospective validation with the complete wearable biosensor ensemble remains a separate deployment requirement.